In [2]:
import nglview as nv
import requests
import pandas as pd
import numpy as np
import io
from Bio.PDB import PDBList, PDBParser
from IPython.display import clear_output
from tqdm import tqdm
from Bio.PDB.SASA import ShrakeRupley

In [2]:
sabdab = '/Users/juliawu/b-cell-epitope/data/sabdab_summary_all.tsv'
df_sabdab = pd.read_csv(sabdab, sep='\t')

df_sabdab['pdb'] = df_sabdab['pdb'].str.upper()
df_sabdab.head()

df_sabdab[df_sabdab['pdb'] == '7UOT']

,pdb,Hchain,Lchain,model,antigen_chain,antigen_type,antigen_het_name,antigen_name,short_header,date,...,scfv,engineered,heavy_subclass,light_subclass,light_ctype,affinity,delta_g,affinity_method,temperature,pmid
7680,7UOT,S,T,0,c,protein,NaN,unknown,VIRAL PROTEIN/IMMUNE SYSTEM,11/02/22,...,False,True,IGHV1,IGKV3,Kappa,NaN,NaN,NaN,NaN,NaN
7681,7UOT,O,P,0,a,protein,NaN,unknown,VIRAL PROTEIN/IMMUNE SYSTEM,11/02/22,...,False,True,IGHV1,IGKV3,Kappa,NaN,NaN,NaN,NaN,NaN
7682,7UOT,Q,R,0,b,protein,NaN,unknown,VIRAL PROTEIN/IMMUNE SYSTEM,11/02/22,...,False,True,IGHV1,IGKV3,Kappa,NaN,NaN,NaN,NaN,NaN
7683,7UOT,H,L,0,C | A | B,protein | protein | protein,NA | NA | NA,glycoprotein g2 | glycoprotein g2 | glycoprote...,VIRAL PROTEIN/IMMUNE SYSTEM,11/02/22,...,False,True,IGHV4,IGLV2,Lambda,NaN,NaN,NaN,NaN,NaN


In [4]:
max_accessibility = {
    'ALA': 106,
    'CYS': 135,
    'ASP': 163,
    'GLU': 194,
    'PHE': 197,
    'GLY': 84,
    'HIS': 184,
    'ILE': 169,
    'LYS': 205,
    'LEU': 164,
    'MET': 188,
    'ASN': 157,
    'PRO': 136,
    'GLN': 198,
    'ARG': 248,
    'SER': 130,
    'THR': 142,
    'VAL': 142,
    'TRP': 227,
    'TYR': 222,
}

backbone_atoms = ['N', 'CA', 'C', 'O', 'H', 'HA', 'HN', 'OXT']
# side chain atoms named by distance from backbone (ex. CB-->  caron beta)

In [28]:
filtered_rows = df_sabdab[df_sabdab['pdb'] == '7UOT']
#print(len(filtered_rows))

class find_chain():
    def __init__(self, pdb_id):
        self.Hchain = []
        self.Lchain = []
        self.Achain = []
        self.name = pdb_id
    def update_chain(self, Hchain, Lchain, Achain):
        self.Hchain.append(Hchain)
        self.Lchain.append(Lchain)
        self.Achain.append(Achain)
    def __str__(self):
        return f'Hchain: {self.Hchain} Lchain: {self.Lchain} Achain: {self.Achain}'

df_trun = df_sabdab.head(2)

#print(df_trun)

chain = find_chain('7UOT')
chain.update_chain('S', 'T', 'c')
print(chain.__str__())

chain.update_chain('a', 'b', 'c')
print(chain.__str__())


print(chain.Hchain[0])

def find_abag_chains(pdb_id):

    chain = find_chain(pdb_id)
    
    filtered_rows = df_sabdab[df_sabdab['pdb'] == pdb_id]
    for index in range(len(filtered_rows)):
        
        Hchain = filtered_rows.iloc[index, 1]
        Lchain = filtered_rows.iloc[index, 2]
        Achain = filtered_rows.iloc[index, 4]

        chain.update_chain(Hchain, Lchain, Achain)
    return chain.__str__()

print(find_abag_chains('7UOT'))


# 7UOT 
# iterate through every chain 
# drop duplicates later in pd 
# can have antibodies binding to dif areas in antigens 
# iterows: index, row --> row parameter, accesses whole row 

Hchain: ['S'] Lchain: ['T'] Achain: ['c']
Hchain: ['S', 'a'] Lchain: ['T', 'b'] Achain: ['c', 'c']
S
Hchain: ['S', 'O', 'Q', 'H'] Lchain: ['T', 'P', 'R', 'L'] Achain: ['c', 'a', 'b', 'C | A | B']


In [5]:
def calculate_epitope(pdb_id, antibody_chains, antigen_chains, cutoff):
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        pdb_text = requests.get(url).text

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, io.StringIO(pdb_text))
        sr = ShrakeRupley()
        sr.compute(structure, level="R")

        if 0 not in structure.child_dict:
            print(f'you\'re cooked, no model in structure bro')
            return pd.DataFrame(), []
        
        model = structure[0]
        # print(f'pdbid {pdb_id} ab chains {antibody_chains} ag chains {antigen_chains}')


        ag_residues = []
        ag_res_counter = 0
        for chain in antigen_chains:
            if chain not in model:
                print(f'bro antigen chain: {antigen_chains} aint in your model: {model}')
                return pd.DataFrame, []
            for res in model[chain].get_residues():
                if res.id[0] == ' ': # res.id is a tuple (residue, seq number)
                    ag_residues.append(res.resname)
                    ag_res_counter += 1

        if ag_res_counter < 30:
            print(f'antigen is probably truncated! its length is {ag_res_counter}')

        ab_info = []
        ab_atoms = []
        for chain in antibody_chains:
            if chain not in model: 
                print(f'bro antigen chain: {antibody_chains} aint in your model: {model}')
                return pd.DataFrame(), []
            for res in model[chain].get_residues():
                ab_info.append((chain, res.resname))
            for atom in model[chain].get_atoms():
                ab_atoms.append(atom.coord)

        results = []
        residues = []

        for chain in antigen_chains:
            for res in model[chain].get_residues():
                #print(f'residue: {res}')

                # skip water and ligands
                if res.id[0] != " ":
                    continue

                # get all atom coords for this residue
                ag_atoms = []
                #ag_atoms_check = [] # to check if the atoms are in the side chain
                for atom in res.get_atoms():
                    ag_atoms.append(atom.coord) 
                    #ag_atoms_check.append(atom)
                ag_atoms = np.array(ag_atoms)
                #print(ag_atoms_check)

                min_dist = 9999
                # calculate distance from each antigen atom to each antibody atom
                for ag_coord in ag_atoms:
                    for ab_coord in ab_atoms:
                        dist = np.sqrt(((ag_coord[0] - ab_coord[0])**2)+((ag_coord[1] - ab_coord[1])**2)+((ag_coord[2] - ab_coord[2])**2)) # the euclidean distance cus it's in x,y,z space
                        #print(f'distance: {d}')
                        #print(f'ag_coord: {ag_coord}')
                        #print(f'ab_coord: {ab_coord}')
                        #print(type(ab_coord), ab_coord.shape
                        # if it's under 5A then it's the epitope!!!
                        if dist < min_dist:
                            min_dist = dist 
                            # dupes... multiple atoms under same residue that're under 5 A, but we only want the residue 
                if min_dist < cutoff:
                    if res.id[1] not in residues:
                        if atom.get_name() not in backbone_atoms: # (expression for variable in iterable if condition)
                            
        
                            residues.append(res.id[1])
                            # print(ag_atoms_check)
                            
                            # checking sasa score: if the residue is solvent accessible 
                            # print(chain)
                            absolute_sasa = res.get_resname(),round(res.sasa,2) # structure, chain, residue, atom, to 2 decimal places
                            res_name = str(res.get_resname())
                            relative_sasa = (absolute_sasa[1] / max_accessibility[res_name]) * 100
                            
                            #print(relative_sasa)
                            if relative_sasa >= 16: #from rost and sander paper

                                results.append({
                                    "chain"        : chain,
                                    "residue number" : res.id[1],
                                    "residue name" : res_name,
                                    "distance (A)" : min_dist
                                })
        # print(f'# of epitopes {len(results)}') # comment on what i think things are as i do them 
        df = pd.DataFrame(results)
        if df.empty == True:
            return df, []
        
        resi_list = df['residue number'].to_list()
        string_resi_list = [str(num) for num in resi_list]
        pymol_list = ('+'.join(string_resi_list))

        return(df, pymol_list)

df, pymol_list = calculate_epitope('9IW2', ['H', 'L'], ['F'], 5)

# GET THE ATOM IN THE SIDE CHAIN instead, not in the backbone..... FIX THIS!!!!

# frequencies in EV files 
# a2m file is the homolog pairs, calculate the mutability from the a2m files anyways 
# my work is ground truth --> sequences (ev coupling)does mutability correspond to epitope at all --> using sequences alone 
# does score and mutability correspond to real epitope

# antibody chain has sep sequence lol!!! add antibody residue epitope as well for santiy check

print(df)
print(pymol_list)

  chain  residue number residue name  distance (A)
0     F              61          LYS      3.249471
61
